# BirdCLEF+ 2026 — SED Inference (Stage 1 EfficientNet-B0 Ensemble)

5-fold ensemble of EfficientNet-B0 SED models trained on all 234 species.

**Pipeline**:
- Load 5× ONNX models from dataset
- Sliding 20s window (5s stride) inference on test soundscapes
- Average ensemble predictions across folds
- Map per-window predictions to row_id → submission CSV

In [ ]:
import os
import math
import time
import warnings
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torchaudio
import torchaudio.transforms as T
import torch
import onnxruntime as ort
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

NOTEBOOK_START = time.time()
BUDGET_SECS    = 85 * 60

# ── Audio / mel parameters (must match training) ───────────────────────────────
SAMPLE_RATE   = 32_000
CHUNK_SAMPLES = 20 * SAMPLE_RATE   # 640 000 samples = 20 s
WIN_STRIDE    = 5  * SAMPLE_RATE   # 5 s stride
N_MELS        = 224
N_FFT         = 4096
HOP_LENGTH    = 1252
F_MIN         = 0
F_MAX         = 16_000
TOP_DB        = 80
N_CLASSES     = 234

# CPU threads
_ncpu = os.cpu_count() or 4
print(f'CPU threads: {_ncpu}')

# ── Mel transform (CPU, reused across all files) ───────────────────────────────
_mel_transform   = T.MelSpectrogram(
    sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH,
    n_mels=N_MELS, f_min=F_MIN, f_max=F_MAX,
    power=2.0, norm='slaney', mel_scale='slaney', center=True,
)
_amplitude_to_db = T.AmplitudeToDB(stype='power', top_db=TOP_DB)


def waveform_to_mel(wav: np.ndarray) -> np.ndarray:
    """(CHUNK_SAMPLES,) → (1, 3, N_MELS, T) float32 numpy."""
    t = torch.from_numpy(wav).float().unsqueeze(0)  # (1, T)
    mel    = _mel_transform(t)
    mel_db = _amplitude_to_db(mel)
    mel_db = mel_db - mel_db.min()
    peak   = mel_db.max()
    if peak > 0:
        mel_db = mel_db / peak
    img = mel_db.repeat(3, 1, 1).unsqueeze(0).numpy()  # (1, 3, N_MELS, T)
    return img


# ── Locate ONNX models ─────────────────────────────────────────────────────────
_onnx_hits = subprocess.run(
    ['find', '/kaggle/input', '-name', '*.onnx', '-not', '-path', '*/perch/*'],
    capture_output=True, text=True
).stdout.strip().splitlines()
_onnx_hits = sorted(x for x in _onnx_hits if x)
print(f'ONNX models found: {len(_onnx_hits)}')
for p in _onnx_hits:
    print(' ', p)

assert _onnx_hits, 'No ONNX models found under /kaggle/input'

# ── Load ONNX sessions ─────────────────────────────────────────────────────────
sess_options = ort.SessionOptions()
sess_options.intra_op_num_threads = _ncpu
sess_options.inter_op_num_threads = 1

sessions = []
for p in _onnx_hits:
    sess = ort.InferenceSession(
        p,
        sess_options=sess_options,
        providers=['CPUExecutionProvider'],
    )
    sessions.append(sess)
    print(f'Loaded: {Path(p).name}')

# Warmup
dummy = np.zeros((1, 3, N_MELS, 512), dtype=np.float32)
for s in sessions:
    s.run(['probs'], {'mel': dummy})
print(f'Warmup done. Ensemble size: {len(sessions)}')

# ── Locate competition data ────────────────────────────────────────────────────
COMP_DATA = None
for p in sorted(Path('/kaggle/input').iterdir()):
    if (p / 'sample_submission.csv').exists():
        COMP_DATA = p
        break
assert COMP_DATA, 'Cannot find sample_submission.csv'
print(f'Competition data: {COMP_DATA}')

sample_sub   = pd.read_csv(COMP_DATA / 'sample_submission.csv')
COMP_SPECIES = list(sample_sub.columns[1:])
assert len(COMP_SPECIES) == N_CLASSES, f'Expected {N_CLASSES} species, got {len(COMP_SPECIES)}'
print(f'Species: {len(COMP_SPECIES)}')

In [ ]:
# ── Inference helpers ──────────────────────────────────────────────────────────

def load_audio(path: Path) -> np.ndarray:
    data, sr = sf.read(str(path), dtype='float32', always_2d=True)
    data = data.mean(axis=1) if data.shape[1] > 1 else data[:, 0]
    if sr != SAMPLE_RATE:
        raise ValueError(f'Expected {SAMPLE_RATE} Hz, got {sr} Hz')
    mx = np.abs(data).max()
    return data / mx if mx > 0 else data


def pad_or_crop(wav: np.ndarray, length: int) -> np.ndarray:
    n = len(wav)
    if n == length:
        return wav
    if n < length:
        reps = -(-length // n)
        wav = np.tile(wav, reps)
    return wav[:length]


def predict_file(waveform: np.ndarray) -> np.ndarray:
    """Sliding 20s window with 5s stride → (n_windows, N_CLASSES) float32."""
    total   = len(waveform)
    n_win   = max(1, math.ceil((total - CHUNK_SAMPLES) / WIN_STRIDE) + 1)
    results = []

    for i in range(n_win):
        start = i * WIN_STRIDE
        end   = start + CHUNK_SAMPLES
        seg   = waveform[start:end]
        seg   = pad_or_crop(seg, CHUNK_SAMPLES)
        mel   = waveform_to_mel(seg)  # (1, 3, N_MELS, T)

        probs_acc = np.zeros(N_CLASSES, dtype=np.float64)
        for sess in sessions:
            probs_acc += sess.run(['probs'], {'mel': mel})[0][0]
        results.append((probs_acc / len(sessions)).astype(np.float32))

    return np.stack(results, axis=0)  # (n_win, N_CLASSES)


def row_ids_for(stem: str, n_win: int) -> list:
    return [f'{stem}_{(i + 1) * 5}' for i in range(n_win)]

In [ ]:
# ── Run inference on all test soundscapes ─────────────────────────────────────
UNIFORM_SCORE = 1.0 / N_CLASSES

test_files = sorted((COMP_DATA / 'test_soundscapes').glob('*.ogg'))
print(f'Test soundscapes: {len(test_files)}')

all_ids   = []
all_probs = []

if not test_files:
    print('No test files — generating uniform-score placeholder submission.')
    for rid in sample_sub['row_id']:
        all_ids.append(rid)
        all_probs.append(np.full((1, N_CLASSES), UNIFORM_SCORE, dtype=np.float32))
else:
    for fpath in tqdm(test_files, desc='Inference'):
        elapsed = time.time() - NOTEBOOK_START
        if elapsed > BUDGET_SECS:
            print(f'\n⚠ Time budget reached ({elapsed/60:.1f} min) — '
                  f'filling remaining files with uniform scores')
            for fp in test_files[len(all_probs):]:
                n_win = 12
                all_ids.extend(row_ids_for(fp.stem, n_win))
                all_probs.append(np.full((n_win, N_CLASSES), UNIFORM_SCORE, dtype=np.float32))
            break

        try:
            waveform = load_audio(fpath)
            probs    = predict_file(waveform)
            all_ids.extend(row_ids_for(fpath.stem, len(probs)))
            all_probs.append(probs)
        except Exception as e:
            print(f'  [warn] {fpath.name}: {e}')
            n_win = 12
            all_ids.extend(row_ids_for(fpath.stem, n_win))
            all_probs.append(np.full((n_win, N_CLASSES), UNIFORM_SCORE, dtype=np.float32))

probs_matrix = np.concatenate(all_probs, axis=0)
print(f'Total windows: {len(probs_matrix)}  |  elapsed: {(time.time()-NOTEBOOK_START)/60:.1f} min')

In [ ]:
# ── Build & save submission ────────────────────────────────────────────────────
sub = pd.DataFrame(probs_matrix, columns=COMP_SPECIES)
sub.insert(0, 'row_id', all_ids)

sub = sample_sub[['row_id']].merge(sub, on='row_id', how='left')
sub[COMP_SPECIES] = sub[COMP_SPECIES].fillna(UNIFORM_SCORE)

print(f'Submission shape: {sub.shape}')
print(f'Missing values:   {sub.isnull().sum().sum()}')
print(sub.head(3))

out_path = Path('/kaggle/working/submission.csv')
sub.to_csv(out_path, index=False)
print(f'\nSaved → {out_path}')